<a href="https://colab.research.google.com/github/fernandouribeu/analisis_datos/blob/main/Laboratorio_2_Notebook_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio Práctico 2 · Pipeline ETL reproducible

**Curso:** Análisis de Datos  
**Programa:** Magíster en Data Science  
**Universidad del Desarrollo**

**Nombre estudiante:**  Fernando Uribe
**Fecha:**  23-08-2026

---

## Contexto del problema

En el Laboratorio Práctico 1 se realizó un diagnóstico inicial de calidad sobre un conjunto de datos de clientes de una empresa de retail.

En este laboratorio se utilizarán esos mismos datos para construir un **pipeline reproducible de extracción, transformación y preparación de datos (ETL)**.

El objetivo no es simplemente corregir valores, sino desarrollar un proceso documentado que permita transformar los datos originales en una versión preparada para análisis posteriores.

Una decisión importante del proceso será distinguir entre:

- errores o problemas de calidad que requieren tratamiento;
- valores faltantes que requieren una estrategia explícita;
- inconsistencias que deben investigarse antes de modificarse;
- valores que pueden parecer extraños, pero que podrían ser válidos.

El resultado final debe poder reproducirse ejecutando nuevamente el notebook desde el comienzo.


## Objetivos del laboratorio

Al finalizar esta actividad, se espera que usted sea capaz de:

1. Implementar un proceso reproducible de carga y transformación de datos.
2. Estandarizar tipos y categorías de variables.
3. Detectar y tratar valores faltantes utilizando criterios justificados.
4. Identificar inconsistencias entre variables y documentar las decisiones adoptadas.
5. Crear variables derivadas cuando aporten valor al análisis posterior.
6. Validar la calidad del dataset antes y después del proceso ETL.
7. Documentar las transformaciones realizadas de manera que otra persona pueda comprender y reproducir el proceso.


## 1. Importación de librerías

Importe las librerías necesarias para desarrollar el pipeline.

Se recomienda utilizar:

- pandas
- numpy
- matplotlib
- seaborn

Puede utilizar otras librerías si son necesarias, pero debe justificar su utilización.


In [1]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb

#Se importaron las librerias pandas, numpy, matplotlib y seaborn

## 2. Extract · Carga de los datos originales

Cargue el archivo `clientes_retail.csv`.

El archivo original debe conservarse sin modificaciones. Todas las transformaciones posteriores deben realizarse sobre una copia del dataset.

Incluya:

- carga reproducible del archivo;
- verificación de cantidad de filas y columnas;
- revisión de los nombres de variables;
- una primera inspección de los datos.

### Importante

No modifique directamente el archivo CSV original.


In [2]:

# Carga de la carpeta a drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# Carga del dataset
df = pd.read_csv("/content/drive/MyDrive/Analisis de Datos/Laboratorio2/clientes_retail.csv")

## 3. Diagnóstico inicial del dataset

Antes de realizar transformaciones, construya una línea base que permita comparar el estado de los datos antes y después del proceso ETL.

Evalúe al menos:

- cantidad de registros;
- cantidad de variables;
- tipos de datos;
- valores faltantes;
- registros duplicados;
- categorías relevantes;
- rangos básicos de variables numéricas.

No es necesario repetir todo el análisis exploratorio realizado en el Laboratorio 1. El foco debe estar en identificar los aspectos que serán relevantes para las transformaciones posteriores.


In [4]:
# Diagnóstico inicial

# cantidad de registros;
# cantidad de variables;
filas, columnas = df.shape
print(f"El dataset tiene {filas} filas y {columnas} columnas.")

El dataset tiene 1020 filas y 15 columnas.


In [5]:
# tipos de datos;
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         1020 non-null   object 
 1   age                 1020 non-null   int64  
 2   gender              1020 non-null   object 
 3   education           969 non-null    object 
 4   marital_status      1020 non-null   object 
 5   income_clp          916 non-null    float64
 6   region              1020 non-null   object 
 7   registration_date   1020 non-null   object 
 8   total_purchases     1020 non-null   int64  
 9   total_spent_clp     1020 non-null   int64  
 10  online_purchases    1020 non-null   int64  
 11  store_purchases     1020 non-null   int64  
 12  complaints          1020 non-null   int64  
 13  satisfaction_score  990 non-null    float64
 14  last_purchase_days  1020 non-null   int64  
dtypes: float64(2), int64(7), object(6)
memory usage: 119.7+

In [6]:
# valores faltantes;
df.isna().sum()

,0
customer_id,0
age,0
gender,0
education,51
marital_status,0
income_clp,104
region,0
registration_date,0
total_purchases,0
total_spent_clp,0


In [7]:
# registros duplicados;
df.duplicated().sum()

np.int64(20)

In [8]:
# detalle de los registros duplicados
df[df.duplicated(keep=False)].sort_values(by='customer_id')

,customer_id,age,gender,education,marital_status,income_clp,region,registration_date,total_purchases,total_spent_clp,online_purchases,store_purchases,complaints,satisfaction_score,last_purchase_days
76,C00077,43,Masculino,Universitaria,Viudo,677817.0,Metropolitana,2025-09-24,9,431912,2,11,2,3.0,296
1011,C00077,43,Masculino,Universitaria,Viudo,677817.0,Metropolitana,2025-09-24,9,431912,2,11,2,3.0,296
136,C00137,150,Femenino,Postgrado,Casado,1152768.0,Metropolitana,2019-10-27,15,190776,7,7,1,4.0,300
1009,C00137,150,Femenino,Postgrado,Casado,1152768.0,Metropolitana,2019-10-27,15,190776,7,7,1,4.0,300
280,C00281,43,Femenino,Postgrado,Viudo,NaN,Araucanía,2021-02-20,20,746418,9,7,0,3.0,107
1016,C00281,43,Femenino,Postgrado,Viudo,NaN,Araucanía,2021-02-20,20,746418,9,7,0,3.0,107
1019,C00320,40,Masculino,Media,Viudo,760088.0,Araucanía,2020-05-23,17,428171,7,9,1,4.0,187
319,C00320,40,Masculino,Media,Viudo,760088.0,Araucanía,2020-05-23,17,428171,7,9,1,4.0,187
411,C00412,28,Hombre,Técnica,Soltero,NaN,Metropolitana,2023-11-18,17,934178,7,10,1,5.0,228
1004,C00412,28,Hombre,Técnica,Soltero,NaN,Metropolitana,2023-11-18,17,934178,7,10,1,5.0,228


In [9]:
df["gender"].value_counts()

,count
gender,
Femenino,495
Masculino,465
Otro,19
Hombre,11
femenino,10
FEMENINO,10
M,10


In [10]:
df["age"].value_counts()

,count
age,
44,47
35,45
41,39
42,37
49,37
...,...
74,1
120,1
65,1


In [11]:
df["region"].value_counts()

,count
region,
Metropolitana,257
Valparaíso,251
Araucanía,245
Biobío,235
Santiago,12
Region Metropolitana,10
RM,10


In [12]:
df["education"].value_counts()

,count
education,
Técnica,346
Universitaria,332
Media,234
Postgrado,57


In [13]:
df['marital_status'].value_counts()


,count
marital_status,
Viudo,282
Divorciado,257
Casado,247
Soltero,234


In [14]:
df.describe().T[['count', 'min', 'mean', 'max']]

,count,min,mean,max
age,1020.0,5.0,4.203725e+01,150.0
income_clp,916.0,-500000.0,1.344718e+06,25000000.0
total_purchases,1020.0,3.0,1.513725e+01,27.0
total_spent_clp,1020.0,-30000.0,1.065511e+06,50000000.0
online_purchases,1020.0,0.0,7.025490e+00,18.0
store_purchases,1020.0,1.0,8.058824e+00,19.0
complaints,1020.0,-1.0,1.103922e+00,50.0
satisfaction_score,990.0,0.0,3.665657e+00,10.0
last_purchase_days,1020.0,2.0,1.858961e+02,364.0


## 4. Plan de transformación

Antes de modificar los datos, defina un plan de transformación.

Construya una tabla o estructura equivalente que documente, para cada problema identificado:

- variable afectada;
- problema detectado;
- tratamiento propuesto;
- justificación;
- impacto esperado.

No se espera que todos los valores extraños sean modificados automáticamente. Una transformación debe estar respaldada por un criterio analítico o de negocio.

### Ejemplo

| Variable | Problema | Tratamiento | Justificación |
|---|---|---|---|
| `registration_date` | Tipo incorrecto | Convertir a fecha | Permitir análisis temporal |
| `gender` | Categorías inconsistentes | Estandarizar | Evitar categorías duplicadas por formato |

Complete la tabla con los problemas que considere relevantes en el dataset.


# Transformación de tipos y formatos

| Variable | Problema | Tratamiento | Justificación | Impacto esperado|
|---|---|---|---|---|
| `df` (global) | Filas duplicadas | Eliminar duplicados | Evitar sobreestimación | Mayor precisión |
| `registration_date` | Formato texto | Convertir a `datetime` | Permitir análisis temporal | Ordenar o filtrar por fecha |
| `gender` | Formatos inconsistentes | Estandarizar | Evitar categorías duplicadas por formato
| Agrupaciones correctas |
| `region`|  Formatos inconsistentes | Estandarizar | Evitar categorías duplicadas por formato
| Agrupaciones correctas |
| `age` | Valores fuera de rango | Filtrar rangos no válidos | Asegurar coherencia | Mejorar medidas |
| `income_clp` | Valores faltantes | Agregar promedio por nivel educacional | Evitar posibles distorciones | Mantener la consistencia |

## 5. Transformación de tipos y formatos

Revise y transforme los tipos de datos que sean necesarios para trabajar correctamente con el dataset.

Considere especialmente:

- fechas;
- variables numéricas;
- variables categóricas;
- identificadores.

Después de realizar las transformaciones, verifique que los tipos obtenidos sean los esperados.

Documente cualquier decisión que no sea evidente.


In [47]:
# Se guardo una versión de respaldo del DataFrame original y se creo una de trabajo
df_original = df.copy()
df_trabajo = df.copy()

In [48]:
# Limpieza de registros duplicados;
df_trabajo = df_trabajo.drop_duplicates().copy()

In [49]:
# Check registros duplicados;
df_trabajo.duplicated().sum()

np.int64(0)

In [50]:
# Convertir a entero preservando valores nulos (NaN)
df_trabajo['income_clp'] = df_trabajo['income_clp'].round().astype('Int64')
df.head()

,customer_id,age,gender,education,marital_status,income_clp,region,registration_date,total_purchases,total_spent_clp,online_purchases,store_purchases,complaints,satisfaction_score,last_purchase_days
0,C00001,47,Femenino,Media,Soltero,781889.0,Biobío,2024-02-05,12,537694,7,5,0,4.0,206
1,C00002,40,Femenino,Media,Soltero,1242577.0,Araucanía,2020-02-11,9,862822,6,11,0,4.0,242
2,C00003,49,Masculino,Media,Soltero,1347579.0,Araucanía,2020-10-12,20,1195473,5,4,1,4.0,306
3,C00004,60,Masculino,Técnica,Casado,686279.0,Biobío,2023-10-13,15,1571007,4,8,0,3.0,184
4,C00005,39,Femenino,Universitaria,Casado,1394730.0,Biobío,2018-01-29,14,410407,7,6,0,3.0,128


In [51]:
# Forzar el formato estandarizado (YYYY-MM-DD)
df_trabajo['registration_date'] = pd.to_datetime(
    df_trabajo['registration_date'],
    format='mixed',
    dayfirst=True
).dt.strftime('%Y-%m-%d')

In [52]:
# Convertir a datetime
df_trabajo['registration_date'] = pd.to_datetime(df_trabajo['registration_date'])
df.head()

,customer_id,age,gender,education,marital_status,income_clp,region,registration_date,total_purchases,total_spent_clp,online_purchases,store_purchases,complaints,satisfaction_score,last_purchase_days
0,C00001,47,Femenino,Media,Soltero,781889.0,Biobío,2024-02-05,12,537694,7,5,0,4.0,206
1,C00002,40,Femenino,Media,Soltero,1242577.0,Araucanía,2020-02-11,9,862822,6,11,0,4.0,242
2,C00003,49,Masculino,Media,Soltero,1347579.0,Araucanía,2020-10-12,20,1195473,5,4,1,4.0,306
3,C00004,60,Masculino,Técnica,Casado,686279.0,Biobío,2023-10-13,15,1571007,4,8,0,3.0,184
4,C00005,39,Femenino,Universitaria,Casado,1394730.0,Biobío,2018-01-29,14,410407,7,6,0,3.0,128


In [53]:
df_trabajo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         1000 non-null   object        
 1   age                 1000 non-null   int64         
 2   gender              1000 non-null   object        
 3   education           950 non-null    object        
 4   marital_status      1000 non-null   object        
 5   income_clp          900 non-null    Int64         
 6   region              1000 non-null   object        
 7   registration_date   1000 non-null   datetime64[ns]
 8   total_purchases     1000 non-null   int64         
 9   total_spent_clp     1000 non-null   int64         
 10  online_purchases    1000 non-null   int64         
 11  store_purchases     1000 non-null   int64         
 12  complaints          1000 non-null   int64         
 13  satisfaction_score  970 non-null    float64       
 14

## 6. Estandarización de variables categóricas

Identifique variables categóricas que presenten diferencias de formato o escritura.

Por ejemplo, una misma categoría podría aparecer utilizando diferencias de mayúsculas o minúsculas.

Realice una estandarización reproducible cuando corresponda.

Después de transformar los datos:

- revise las categorías resultantes;
- compruebe que no se hayan creado categorías artificiales;
- documente las reglas utilizadas.


In [54]:
# Estandarización de valores en genero a minusculas
df_trabajo['gender'] = df_trabajo['gender'].str.lower()

# Generar dicionario para otros valores como "hombre" o "m" a "masculino"
gender_dict = {
    'hombre': 'masculino',
    'm': 'masculino',
    'mujer': 'femenino',
    'f': 'femenino'
}
# Corregir valores segun diccionario
df_trabajo['gender'] = df_trabajo['gender'].replace(gender_dict)
df_trabajo['gender'].value_counts()

,count
gender,
femenino,505
masculino,476
otro,19


In [55]:
# Estandarización de valores en region a minusculas
df_trabajo['region'] = df_trabajo['region'].str.lower()

# Generar dicionario para otros valores como "region metropolitana" o "rm" a "metropolitana"
region_dict = {
    'region metropolitana': 'metropolitana',
    'rm': 'metropolitana',
    'santiago': 'metropolitana'
}

# Corregir valores segun diccionario
df_trabajo['region'] = df_trabajo['region'].replace(region_dict)
df_trabajo['region'].value_counts()

,count
region,
metropolitana,281
valparaíso,250
araucanía,239
biobío,230


## 7. Valores faltantes

Identifique las variables que presentan valores faltantes y determine una estrategia de tratamiento para cada caso relevante.

Las estrategias pueden incluir, dependiendo del contexto:

- imputación;
- conservación del valor faltante;
- eliminación de registros;
- creación de indicadores de ausencia;
- utilización de reglas específicas de negocio.

No utilice una única estrategia para todas las variables sin justificarla.

Para cada decisión explique:

- por qué eligió esa estrategia;
- qué supuestos está realizando;
- qué posibles efectos puede tener sobre análisis posteriores.


In [56]:
# Estandarización de variables categóricas
# Quitar valores fuera de rango en edad, fijando como rango valido de 18 a 90 años reduciendo a 997 filas
df_trabajo = df_trabajo[(df_trabajo['age'] >= 18) & (df_trabajo['age'] <= 90)].copy()
df_trabajo['age'].value_counts()

,count
age,
44,46
35,44
41,39
42,37
49,36
46,35
36,34
39,34
45,34


In [57]:
# Tratamiento de valores faltantes
# Calcular valor promedio de ingreso categorizando a los clientes por nidel de educación
income_mean = df_trabajo.groupby('education')['income_clp'].mean()
income_mean

,income_clp
education,
Media,1414514.292453
Postgrado,1456335.729167
Técnica,1288594.157718
Universitaria,1349773.133106


In [59]:
# Reemplazar valores NaN en income por la media calculada en los casos que se sabe la education del cleinte
# Redondear y convertir el mapeo a Int64 antes de rellenar
valores_imputacion = df_trabajo['education'].map(income_mean).round().astype('Int64')

# Se completan los campos NAN
df_trabajo['income_clp'] = df_trabajo['income_clp'].fillna(valores_imputacion)

In [60]:
df_trabajo['income_clp'].value_counts(dropna=False).sort_values(ascending=False)

,count
income_clp,
1288594,42
1349773,31
1414514,17
1456336,6
<NA>,3
...,...
561125,1
970690,1
493294,1


In [61]:
# Completar datos de satifación usando la meadiana de los valores
mediana_satisfaccion = df_trabajo['satisfaction_score'].median()
df_trabajo['satisfaction_score'] = df_trabajo['satisfaction_score'].fillna(mediana_satisfaccion)

In [62]:
df_trabajo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 997 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         997 non-null    object        
 1   age                 997 non-null    int64         
 2   gender              997 non-null    object        
 3   education           947 non-null    object        
 4   marital_status      997 non-null    object        
 5   income_clp          994 non-null    Int64         
 6   region              997 non-null    object        
 7   registration_date   997 non-null    datetime64[ns]
 8   total_purchases     997 non-null    int64         
 9   total_spent_clp     997 non-null    int64         
 10  online_purchases    997 non-null    int64         
 11  store_purchases     997 non-null    int64         
 12  complaints          997 non-null    int64         
 13  satisfaction_score  997 non-null    float64       
 14 

In [63]:
df_trabajo[df_trabajo['education'].isna()]

,customer_id,age,gender,education,marital_status,income_clp,region,registration_date,total_purchases,total_spent_clp,online_purchases,store_purchases,complaints,satisfaction_score,last_purchase_days
31,C00032,64,masculino,NaN,Divorciado,1007515,valparaíso,2020-08-27,10,604457,5,8,0,4.0,2
39,C00040,44,masculino,NaN,Viudo,771274,valparaíso,2025-06-16,16,601612,12,6,3,3.0,167
75,C00076,51,femenino,NaN,Divorciado,1638935,biobío,2019-04-01,16,638822,3,6,1,4.0,114
81,C00082,46,femenino,NaN,Viudo,2719503,metropolitana,2025-01-24,18,1294820,5,8,1,3.0,36
84,C00085,32,femenino,NaN,Divorciado,1333267,valparaíso,2020-10-10,18,815661,7,7,1,4.0,272
90,C00091,43,masculino,NaN,Soltero,1103725,metropolitana,2024-04-30,19,1041920,7,9,0,5.0,283
156,C00157,64,masculino,NaN,Divorciado,1752799,metropolitana,2025-03-09,14,1783312,4,5,1,5.0,253
159,C00160,49,masculino,NaN,Viudo,1013745,valparaíso,2023-12-05,8,1872198,8,5,0,1.0,2
165,C00166,46,femenino,NaN,Casado,1381617,valparaíso,2025-11-10,20,270726,10,3,0,2.0,94
172,C00173,41,masculino,NaN,Soltero,1006879,metropolitana,2025-12-02,14,435331,7,13,2,5.0,354


In [64]:
# Definir la regla de asignación por cercanía reutilizando tu 'income_mean'
def asignación_por_cercania(ingreso):
    if pd.isna(ingreso):
        return None
    # Encuentra la categoría cuya media esté más cerca del ingreso del cliente
    return (income_mean - ingreso).abs().idxmin()

# Máscara para identificar las filas con educación nula
mask_edu_null = df_trabajo['education'].isna()

# Imputar las filas que sí tienen ingreso
df_trabajo.loc[mask_edu_null, 'education'] = df_trabajo.loc[mask_edu_null, 'income_clp'].apply(asignación_por_cercania)


In [65]:
df_trabajo.info()


<class 'pandas.core.frame.DataFrame'>
Index: 997 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         997 non-null    object        
 1   age                 997 non-null    int64         
 2   gender              997 non-null    object        
 3   education           994 non-null    object        
 4   marital_status      997 non-null    object        
 5   income_clp          994 non-null    Int64         
 6   region              997 non-null    object        
 7   registration_date   997 non-null    datetime64[ns]
 8   total_purchases     997 non-null    int64         
 9   total_spent_clp     997 non-null    int64         
 10  online_purchases    997 non-null    int64         
 11  store_purchases     997 non-null    int64         
 12  complaints          997 non-null    int64         
 13  satisfaction_score  997 non-null    float64       
 14 

In [66]:
# Para poder estudiar d emejor manera las relaciones eliminare las 3 filas con Nan
df_trabajo = df_trabajo.dropna(subset=['education', 'income_clp'], how='all')

In [67]:
df_trabajo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         994 non-null    object        
 1   age                 994 non-null    int64         
 2   gender              994 non-null    object        
 3   education           994 non-null    object        
 4   marital_status      994 non-null    object        
 5   income_clp          994 non-null    Int64         
 6   region              994 non-null    object        
 7   registration_date   994 non-null    datetime64[ns]
 8   total_purchases     994 non-null    int64         
 9   total_spent_clp     994 non-null    int64         
 10  online_purchases    994 non-null    int64         
 11  store_purchases     994 non-null    int64         
 12  complaints          994 non-null    int64         
 13  satisfaction_score  994 non-null    float64       
 14 

## 8. Reglas de consistencia y validación

Identifique relaciones entre variables que permitan detectar posibles inconsistencias.

Por ejemplo, considere la relación entre:

- `total_purchases`;
- `online_purchases`;
- `store_purchases`.

Evalúe si existe una regla de consistencia que pueda aplicarse y determine qué hacer cuando dicha regla no se cumple.

### Importante

No asuma automáticamente que una diferencia entre variables representa un error. Considere las posibles definiciones de negocio y documente sus supuestos.

Proponga al menos una regla de validación adicional que considere relevante para este dataset.


In [95]:
# Reglas de consistencia y validación
# Definir la suma de compras por canal
df_trabajo = df_trabajo.copy()
df_trabajo['sum_channels'] = df_trabajo['online_purchases'] + df_trabajo['store_purchases']

# Identificar registros donde existen diferencias
diferencias = df_trabajo[df_trabajo['total_purchases'] != df_trabajo['sum_channels']]
print(f"Total de registros con diferencias: {len(diferencias)}")


Total de registros con diferencias: 929


In [96]:
compras_mayores = (df_trabajo['total_purchases'] > df_trabajo['sum_channels']).sum()
compras_menores = (df_trabajo['total_purchases'] < df_trabajo['sum_channels']).sum()

print(f"Casos donde total_purchases es MAYOR que la suma: {compras_mayores}")
print(f"Casos donde total_purchases es MENOR que la suma: {compras_menores}")

Casos donde total_purchases es MAYOR que la suma: 473
Casos donde total_purchases es MENOR que la suma: 456


Las diferencia entre la suma de las compras Online y la tienda contra el total de compras muestra un gran numero de diferencias, cuando el total supera a los canales directos, esto puede responder a canales adicionales de venta(como una App). Por otro lado, cuando el total resulta menor que la suma de las partes, se trata de una inconsistencia puede deberse a problemas de sincorinización por ejemplo compras que no terminan de ser incorporadas.

## 9. Valores extremos y decisiones de transformación

Revise las variables numéricas relevantes e identifique posibles valores extremos.

No es obligatorio eliminar outliers.

Para los casos identificados, determine si corresponde:

- mantenerlos;
- transformarlos;
- marcarlos mediante una variable indicadora;
- excluirlos bajo una regla explícita.

Justifique la decisión adoptada.

El objetivo es demostrar criterio analítico, no eliminar automáticamente todos los valores extremos.


In [97]:
df_trabajo.describe().T[['count', 'min', 'mean', 'max']]

,count,min,mean,max
age,994.0,18.0,41.775654,85.0
income_clp,994.0,0.0,1349067.707243,25000000.0
registration_date,994,2018-01-13 00:00:00,2021-12-11 21:22:05.553319936,2025-12-30 00:00:00
total_purchases,994.0,3.0,15.137827,27.0
total_spent_clp,994.0,15000.0,1022475.911469,50000000.0
online_purchases,994.0,0.0,7.014085,18.0
store_purchases,994.0,1.0,8.041247,19.0
complaints,994.0,0.0,1.104628,50.0
satisfaction_score,994.0,0.0,3.671026,10.0
last_purchase_days,994.0,2.0,185.686117,364.0


In [79]:
# Seleccionar columnas numéricas
cols_numericas = df_trabajo.select_dtypes(include=['int64', 'Int64', 'float64']).columns

# Auditar negativos
resumen_negativos = []

for col in cols_numericas:
    conteo_neg = (df_trabajo[col] < 0).sum()
    if conteo_neg > 0:
        minimo = df_trabajo[col].min()
        resumen_negativos.append({'Variable': col, 'Casos Negativos': conteo_neg, 'Valor Mínimo': minimo})

# Mostrar resultado
df_negativos = pd.DataFrame(resumen_negativos)
print(df_negativos)

          Variable  Casos Negativos  Valor Mínimo
0       income_clp                1       -500000
1  total_spent_clp                2        -30000
2       complaints                1            -1


In [91]:
# Se asumen lso valores negativos como error de digitación y se convierten
# Asegurar copia independiente
df_trabajo = df_trabajo.copy()

# Convertir a valor absoluto las 3 variables afectadas
cols_con_negativos = ['income_clp', 'total_spent_clp', 'complaints']

for col in cols_con_negativos:
    df_trabajo[col] = df_trabajo[col].abs()

# Comprobar que ya no existan valores menores a 0
print("Negativos restantes:", (df_trabajo[cols_con_negativos] < 0).sum().sum())

Negativos restantes: 0


## 10. Transformación e ingeniería de variables

Construya al menos **dos variables derivadas** que puedan aportar valor para análisis posteriores.

Las variables deben construirse utilizando información disponible en el dataset y deben tener una interpretación clara.

Para cada variable creada indique:

- nombre;
- fórmula o regla utilizada;
- interpretación;
- utilidad potencial para análisis posteriores.

Ejemplos posibles incluyen indicadores relacionados con comportamiento de compra, antigüedad del cliente o intensidad de uso de canales. No es necesario utilizar estos ejemplos exactamente.


In [93]:
# Ingeniería de variables
# Gasto promedio por compra
df_trabajo['avg_purch_clp'] = (
    df_trabajo['total_spent_clp']
    .div(df_trabajo['total_purchases'])
    .fillna(0)
    .round()
    .astype('Int64')
)

df_trabajo.describe().T[['count', 'min', 'mean', 'max']]


,count,min,mean,max
age,994.0,18.0,41.775654,85.0
income_clp,994.0,0.0,1349067.707243,25000000.0
registration_date,994,2018-01-13 00:00:00,2021-12-11 21:22:05.553319936,2025-12-30 00:00:00
total_purchases,994.0,3.0,15.137827,27.0
total_spent_clp,994.0,15000.0,1022475.911469,50000000.0
online_purchases,994.0,0.0,7.014085,18.0
store_purchases,994.0,1.0,8.041247,19.0
complaints,994.0,0.0,1.104628,50.0
satisfaction_score,994.0,0.0,3.671026,10.0
last_purchase_days,994.0,2.0,185.686117,364.0


In [99]:
# % de compras Online
df_trabajo['%purch_online'] = (
    df_trabajo['online_purchases']
    .div(df_trabajo['total_purchases'])
    .fillna(0.0)
    .round(4)
)

df_trabajo.describe().T[['count', 'min', 'mean', 'max']]


,count,min,mean,max
age,994.0,18.0,41.775654,85.0
income_clp,994.0,0.0,1349067.707243,25000000.0
registration_date,994,2018-01-13 00:00:00,2021-12-11 21:22:05.553319936,2025-12-30 00:00:00
total_purchases,994.0,3.0,15.137827,27.0
total_spent_clp,994.0,15000.0,1022475.911469,50000000.0
online_purchases,994.0,0.0,7.014085,18.0
store_purchases,994.0,1.0,8.041247,19.0
complaints,994.0,0.0,1.104628,50.0
satisfaction_score,994.0,0.0,3.671026,10.0
last_purchase_days,994.0,2.0,185.686117,364.0


Gasto promedio (avg_purch_clp)

Utilidad: Identifica el gasto medio por compra para diseñar promociones y perfilar a clientes.

% Compra Online (%purch_online)

Utilidad: Nos da una idea de como se ubica el uso del canal de compras online contra el total de compras a la tienda.

## 11. Validación del dataset transformado

Una vez finalizadas las transformaciones, realice un nuevo diagnóstico de calidad.

Compare el estado del dataset antes y después del proceso ETL.

Verifique al menos:

- cantidad de registros;
- cantidad de variables;
- tipos de datos;
- valores faltantes;
- duplicados;
- categorías estandarizadas;
- reglas de consistencia;
- variables derivadas.

La validación debe demostrar mediante evidencia que el pipeline produjo el resultado esperado.


In [109]:
# Validación final del dataset
# cantidad de registros
# cantidad de variables
filas, columnas = df_trabajo.shape
print(f"El dataset tiene {filas} filas y {columnas} columnas.")

El dataset tiene 994 filas y 18 columnas.


In [110]:
# tipos de datos
# valores faltantes
# variables derivadas
df_trabajo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, 0 to 999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         994 non-null    object        
 1   age                 994 non-null    int64         
 2   gender              994 non-null    object        
 3   education           994 non-null    object        
 4   marital_status      994 non-null    object        
 5   income_clp          994 non-null    Int64         
 6   region              994 non-null    object        
 7   registration_date   994 non-null    datetime64[ns]
 8   total_purchases     994 non-null    int64         
 9   total_spent_clp     994 non-null    int64         
 10  online_purchases    994 non-null    int64         
 11  store_purchases     994 non-null    int64         
 12  complaints          994 non-null    int64         
 13  satisfaction_score  994 non-null    float64       
 14 

In [111]:
# duplicados
df_trabajo.duplicated().sum()

np.int64(0)

In [113]:
# categorías estandarizadas
df_trabajo["gender"].value_counts()

,count
gender,
femenino,501
masculino,474
otro,19


In [114]:
df_trabajo["region"].value_counts()

,count
region,
metropolitana,279
valparaíso,248
araucanía,238
biobío,229


In [115]:
df_trabajo["education"].value_counts()

,count
education,
Técnica,367
Universitaria,327
Media,232
Postgrado,68


In [116]:
df_trabajo["marital_status"].value_counts()

,count
marital_status,
Viudo,270
Divorciado,256
Casado,239
Soltero,229


## 12. Resumen del pipeline ETL

Documente el flujo completo desarrollado.

Puede representarlo mediante una lista, tabla o diagrama simple.

El resumen debe permitir responder:

1. ¿Qué datos se cargaron?
2. ¿Qué problemas fueron identificados?
3. ¿Qué transformaciones fueron aplicadas?
4. ¿Qué decisiones requirieron criterios o supuestos?
5. ¿Cómo se validó el resultado?
6. ¿Qué dataset queda disponible para análisis posteriores?


## Resumen del Pipeline ETL

1.  **¿Qué datos se cargaron?**
    *   Se cargó el archivo `clientes_retail.csv` con 1020 filas y 15 columnas. Una copia de este `DataFrame` (`df_trabajo`) fue utilizada para todas las transformaciones, conservando el original (`df`) intacto.

2.  **¿Qué problemas fueron identificados?**
    *   **Filas duplicadas:** 20 registros duplicados.
    *   **Tipos de datos incorrectos:** `registration_date` como `object` (cadena de texto) y `income_clp` como `float64` con la intención de convertirlo a `Int64` para manejar `NaN`.
    *   **Categorías inconsistentes:** `gender` (`Femenino`, `Masculino`, `Otro`, `Hombre`, `femenino`, `FEMENINO`, `M`) y `region` (`Metropolitana`, `Valparaíso`, `Araucanía`, `Biobío`, `Santiago`, `Region Metropolitana`, `RM`).
    *   **Valores faltantes:** En `education` (51), `income_clp` (104) y `satisfaction_score` (30).
    *   **Valores fuera de rango (outliers):** `age` (valores < 18 y > 90) y valores negativos en `income_clp`, `total_spent_clp`, y `complaints`.
    *   **Inconsistencias:** Discrepancia entre `total_purchases` y la suma de `online_purchases` + `store_purchases`.

3.  **¿Qué transformaciones fueron aplicadas?**
    *   **Eliminación de duplicados:** Se eliminaron 20 filas duplicadas.
    *   **Conversión de tipos:** `registration_date` se convirtió a `datetime` y `income_clp` a `Int64`.
    *   **Estandarización de categorías:**
        *   `gender`: se convirtieron todos a minúsculas y `hombre`, `m` se mapearon a `masculino`, `mujer`, `f` a `femenino`.
        *   `region`: se convirtieron todos a minúsculas y `region metropolitana`, `rm`, `santiago` se mapearon a `metropolitana`.
    *   **Tratamiento de valores faltantes:**
        *   `age`: Se filtraron registros con `age` fuera del rango de 18 a 90 años (se eliminaron 3 filas).
        *   `income_clp` faltante: Se imputó utilizando la media de `income_clp` agrupada por `education`.
        *   `satisfaction_score` faltante: Se imputó utilizando la mediana de la columna.
        *   `education` y `income_clp` (NaN remanentes): Se eliminaron las 3 filas restantes con valores nulos en ambas columnas.
    *   **Corrección de valores negativos:** Los valores negativos en `income_clp`, `total_spent_clp` y `complaints` se convirtieron a su valor absoluto.
    *   **Ingeniería de variables:**
        *   Se creó `sum_channels` como la suma de `online_purchases` y `store_purchases`.
        *   Se creó `avg_purch_clp` (gasto promedio por compra) = `total_spent_clp` / `total_purchases`.
        *   Se creó `%purch_online` (porcentaje de compras online) = `online_purchases` / `total_purchases`.

4.  **¿Qué decisiones requirieron criterios o supuestos?**
    *   **Rango de edad:** Se asumió que edades fuera de 18-90 años eran errores y se eliminaron. Esto podría requerir validación con el negocio.
    *   **Imputación de ingresos por educación:** Se asumió que el nivel educativo es un buen predictor del ingreso, y que la media de ingresos por nivel educativo es una imputación razonable.
    *   **Imputación de satisfacción por mediana:** Se asumió que la mediana es una medida robusta para imputar valores faltantes en `satisfaction_score`.
    *   **Inconsistencia en `total_purchases`:** Se interpretó que `total_purchases` mayor que `sum_channels` podría deberse a otros canales de venta (como una App), y `total_purchases` menor que `sum_channels` como una inconsistencia de sincronización. No se realizó ninguna corrección automática, solo se identificó el problema.
    *   **Valores negativos:** Se asumió que los valores negativos en `income_clp`, `total_spent_clp` y `complaints` eran errores de digitación y se convirtieron a valores absolutos.

5.  **¿Cómo se validó el resultado?**
    *   Se verificó la cantidad de filas y columnas finales (`994` filas y `18` columnas).
    *   Se inspeccionaron los tipos de datos finales para asegurar que fueran correctos.
    *   Se comprobó la ausencia de valores faltantes en las columnas tratadas (`education`, `income_clp`, `satisfaction_score`).
    *   Se confirmó que no existían registros duplicados.
    *   Se revisaron las `value_counts()` para `gender`, `region`, `education` y `marital_status` para asegurar la estandarización de categorías.

6.  **¿Qué dataset queda disponible para análisis posteriores?**
    *   El `DataFrame` `df_trabajo`, con 994 filas y 18 columnas, con tipos de datos correctos, categorías estandarizadas, valores faltantes tratados y nuevas variables derivadas listas para el análisis.

## 13. Limitaciones y decisiones pendientes

Identifique aspectos del dataset que no hayan podido resolverse completamente o que requieran información adicional del negocio.

Considere especialmente:

- supuestos realizados;
- información que sería necesario solicitar a la empresa;
- transformaciones que podrían cambiar según el objetivo del análisis;
- problemas de calidad que decidiste mantener sin modificar.


## 14. Conclusiones

Explique brevemente el resultado del proceso ETL.

Responda:

- ¿El dataset quedó preparado para análisis posteriores?
- ¿Cuáles fueron las transformaciones más importantes?
- ¿Qué decisiones tuvieron mayor impacto sobre los datos?
- ¿Qué riesgos deberían tenerse presentes al utilizar el dataset transformado?


## Declaración de uso de IA generativa

Complete esta sección si utilizó herramientas de inteligencia artificial generativa durante el desarrollo del laboratorio.

**Herramienta utilizada:**Gemini

**Propósito del uso:**Consulta sobre codigos de Python

**Forma en que validé y adapté la información obtenida:**Adapte el codigo a lo que queria realizar.

**Partes del análisis o código en las que utilicé asistencia de IA:**Para codificar los calculos que necesitaba
